In [2]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# ── CONFIGURATION — ONLY CHANGE THESE ───────────────
NUM_ORDERS = 50000
END_DATE   = "2020-12-31"

# ── LOAD EXISTING FILES ──────────────────────────────
orders        = pd.read_csv("/content/orders.csv",        encoding="latin-1")
order_details = pd.read_csv("/content/order_details.csv", encoding="latin-1")
customers     = pd.read_csv("/content/customers.csv",     encoding="latin-1")
employees     = pd.read_csv("/content/employees.csv",     encoding="latin-1")
shippers      = pd.read_csv("/content/shippers.csv",      encoding="latin-1")
products      = pd.read_csv("/content/products.csv",      encoding="latin-1")

print("All files fetched")

All files fetched


In [3]:
# ── EXTRACT VALID IDs ────────────────────────────────
customer_ids  = customers["customerID"].tolist()
employee_ids  = employees["employeeID"].tolist()
shipper_ids   = shippers["shipperID"].tolist()
active_products = products[products["discontinued"] == 0][["productID", "unitPrice"]]

# ── FIND START POINT ─────────────────────────────────
last_order_id   = orders["orderID"].max()
last_order_date = pd.to_datetime(orders["orderDate"].max())
start_order_id  = last_order_id + 1
start_date      = last_order_date + timedelta(days=1)
end_date        = pd.to_datetime(END_DATE)

print(f"Starting orderID : {start_order_id}")
print(f"Starting date    : {start_date.date()}")
print(f"Ending date      : {end_date.date()}")

# ── SEASONAL WEIGHTS ─────────────────────────────────
# Higher weight = more orders that month
seasonal_weights = {
    1:  0.06,   # Jan — slow
    2:  0.06,   # Feb — slow
    3:  0.07,   # Mar — picking up
    4:  0.08,   # Apr
    5:  0.08,   # May
    6:  0.09,   # Jun — summer peak
    7:  0.10,   # Jul — summer peak
    8:  0.09,   # Aug
    9:  0.08,   # Sep
    10: 0.09,   # Oct — holiday buildup
    11: 0.10,   # Nov — holiday peak
    12: 0.10,   # Dec — holiday peak
}

# ── GENERATE DATE POOL ───────────────────────────────
print("Generating date pool...")
date_pool = []
current = start_date
while current <= end_date:
    weight = seasonal_weights[current.month]
    date_pool.append((current, weight))
    current += timedelta(days=1)

dates   = [d[0] for d in date_pool]
weights = [d[1] for d in date_pool]

# Normalize weights
total   = sum(weights)
weights = [w / total for w in weights]

# Sample order dates
order_dates = np.random.choice(
    range(len(dates)),
    size=NUM_ORDERS,
    p=weights,
    replace=True
)
order_dates = [dates[i] for i in order_dates]
order_dates = sorted(order_dates)  # chronological order

print(f"Date pool generated — {len(dates)} days")

# ── DISCOUNT VALUES (matching original data) ─────────
discount_values  = [0.0, 0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20, 0.25]
discount_weights = [0.50, 0.10, 0.10, 0.05, 0.08, 0.07, 0.05, 0.03, 0.02]

# ── GENERATE ORDERS ──────────────────────────────────
print("Generating orders...")
new_orders = []

for i in range(NUM_ORDERS):
    order_id    = start_order_id + i
    order_date  = order_dates[i]
    required_dt = order_date + timedelta(days=random.randint(7, 30))

    # 95% of orders are shipped, 5% pending
    if random.random() < 0.95:
        shipped_dt = order_date + timedelta(days=random.randint(1, 15))
        shipped_dt = shipped_dt.strftime("%Y-%m-%d")
    else:
        shipped_dt = None

    new_orders.append({
        "orderID":      order_id,
        "customerID":   random.choice(customer_ids),
        "employeeID":   random.choice(employee_ids),
        "orderDate":    order_date.strftime("%Y-%m-%d"),
        "requiredDate": required_dt.strftime("%Y-%m-%d"),
        "shippedDate":  shipped_dt,
        "shipperID":    random.choice(shipper_ids),
        "freight":      round(random.uniform(0.5, 500.0), 2)
    })

df_new_orders = pd.DataFrame(new_orders)
print(f"Orders generated — {len(df_new_orders)} rows")

# ── GENERATE ORDER DETAILS ───────────────────────────
print("Generating order details...")
new_order_details = []

for i in range(NUM_ORDERS):
    order_id = start_order_id + i

    # Number of line items per order (1-6, avg ~2.6)
    num_items = random.choices(
        [1, 2, 3, 4, 5, 6],
        weights=[0.20, 0.30, 0.25, 0.15, 0.07, 0.03]
    )[0]

    # Pick random unique products for this order
    selected_products = active_products.sample(n=min(num_items, len(active_products)))

    for _, product in selected_products.iterrows():
        # Small price variation from base price (±10%)
        base_price = product["unitPrice"]
        unit_price = round(base_price * random.uniform(0.90, 1.10), 2)

        new_order_details.append({
            "orderID":   order_id,
            "productID": int(product["productID"]),
            "unitPrice": unit_price,
            "quantity":  random.randint(1, 100),
            "discount":  random.choices(
                            discount_values,
                            weights=discount_weights
                         )[0]
        })

df_new_order_details = pd.DataFrame(new_order_details)
print(f"Order details generated — {len(df_new_order_details)} rows")

# ── VALIDATE ─────────────────────────────────────────
print("\n── VALIDATION ──────────────────────────────")
print(f"Orders         : {len(df_new_orders)}")
print(f"Order details  : {len(df_new_order_details)}")
print(f"Avg items/order: {round(len(df_new_order_details) / len(df_new_orders), 2)}")
print(f"Date range     : {df_new_orders['orderDate'].min()} to {df_new_orders['orderDate'].max()}")
print(f"OrderID range  : {df_new_orders['orderID'].min()} to {df_new_orders['orderID'].max()}")
print(f"Null shipments : {df_new_orders['shippedDate'].isna().sum()}")

# ── SAVE TO CSV ──────────────────────────────────────
df_new_orders.to_csv("new_orders.csv", index=False)
df_new_order_details.to_csv("new_order_details.csv", index=False)

print("Files saved")

# ── DOWNLOAD FROM COLAB ──────────────────────────────
from google.colab import files
files.download("new_orders.csv")
files.download("new_order_details.csv")

print("Downloads started!")

Starting orderID : 11078
Starting date    : 2015-05-07
Ending date      : 2020-12-31
Generating date pool...
✅ Date pool generated — 2066 days
Generating orders...
Orders generated — 50000 rows
Generating order details...
Order details generated — 134317 rows

── VALIDATION ──────────────────────────────
Orders         : 50000
Order details  : 134317
Avg items/order: 2.69
Date range     : 2015-05-07 to 2020-12-31
OrderID range  : 11078 to 61077
Null shipments : 2500
Files saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started!
